# TurboQuant x Refusal Eval (Kaggle)

This notebook runs baseline vs TurboQuant-style KV cache configurations and reports refusal rate + KL drift.

## Notebook Map (Read First)

This notebook now has one primary result path and two supplemental checks.

1. Setup and auth: Cells 2-6.
2. Primary analysis (latest 3B run): Cells 11-13.
3. Supplemental validations: Cells 7-10 (K/V asymmetry and needle retrieval).

If you only want the core benchmark conclusion, read Cells 11-13 and the interpretation cell immediately after Cell 13.

## 1) Clone and install

If you already attached the repo as a Kaggle Dataset, update paths accordingly.

In [1]:
!nvidia-smi

Sat Apr 11 16:24:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from pathlib import Path

REPO_URL = 'https://github.com/aaliyan1230/turboquant-heretic-kv-bench.git'
REPO_NAME = 'turboquant-heretic-kv-bench'

workdir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo_dir = workdir / REPO_NAME

if not repo_dir.exists():
    !git clone {REPO_URL} {repo_dir}
else:
    print(f'Repo already exists at {repo_dir}')
    !git -C {repo_dir} pull --ff-only

%cd {repo_dir}
!pip -q install -e .

Cloning into '/kaggle/working/turboquant-heretic-kv-bench'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 82 (delta 36), reused 70 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 46.92 KiB | 6.70 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/kaggle/working/turboquant-heretic-kv-bench
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
  Building editable for tqhk (pyproject.toml) ... done


In [3]:
import sys
import importlib
from pathlib import Path

def find_repo_root() -> Path:
    # Try current directory tree first.
    cwd = Path.cwd()
    for base in [cwd, *cwd.parents]:
        if (base / 'src' / 'tqhk').exists():
            return base

    # Common Kaggle locations.
    candidates = [Path('/kaggle/working'), Path('/kaggle/input')]
    for root in candidates:
        if not root.exists():
            continue
        if (root / 'src' / 'tqhk').exists():
            return root
        for child in root.iterdir():
            if child.is_dir() and (child / 'src' / 'tqhk').exists():
                return child

    raise FileNotFoundError(
        "Could not locate repo root containing src/tqhk. "
        "Run the clone/install cell first or set REPO_ROOT manually."
    )

REPO_ROOT = find_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Using REPO_ROOT={REPO_ROOT}')

# Force reload after git pull so benchmark picks up latest source changes.
import tqhk.evaluation as evaluation_mod
import tqhk.cache as cache_mod
import tqhk.benchmark as benchmark_mod
importlib.reload(evaluation_mod)
importlib.reload(cache_mod)
importlib.reload(benchmark_mod)

BenchmarkConfig = benchmark_mod.BenchmarkConfig
RunConfig = benchmark_mod.RunConfig
run_benchmark = benchmark_mod.run_benchmark
CacheConfig = cache_mod.CacheConfig

Using REPO_ROOT=/kaggle/working/turboquant-heretic-kv-bench


In [30]:
import os
import getpass

from huggingface_hub import login

token = getpass.getpass('Enter HF_TOKEN (leave blank to skip): ').strip()
if token:
    os.environ['HF_TOKEN'] = token
    login(token=token, add_to_git_credential=False)
    print('HF login successful.')
else:
    print('No token entered; continuing unauthenticated.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login successful.


In [32]:
import gc
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from tqhk.prompting import build_long_context_prompt

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
DEVICE = 'cuda'
TARGET_CTX_TOKENS = 2048

prompt = build_long_context_prompt(
    user_prompt='Summarize the context in one sentence.',
    filler_repetitions=64,
    system_prompt='You are a helpful assistant.',
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.float16,
    attn_implementation='eager',
)
model.eval()

inputs = tokenizer(
    prompt,
    return_tensors='pt',
    truncation=True,
    max_length=TARGET_CTX_TOKENS,
    return_token_type_ids=False,
).to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs, use_cache=True, return_dict=True)

cache = outputs.past_key_values
layers = cache.layers if hasattr(cache, 'layers') else cache

k_norms = []
v_norms = []
per_layer_rows = []

for li, layer in enumerate(layers):
    if hasattr(layer, 'keys') and hasattr(layer, 'values'):
        k = layer.keys.float()
        v = layer.values.float()
    else:
        k, v = layer[0].float(), layer[1].float()

    k_l2 = torch.linalg.norm(k, dim=-1).detach().cpu().reshape(-1).numpy()
    v_l2 = torch.linalg.norm(v, dim=-1).detach().cpu().reshape(-1).numpy()

    k_finite = k_l2[np.isfinite(k_l2)]
    v_finite = v_l2[np.isfinite(v_l2)]

    if k_finite.size == 0 or v_finite.size == 0:
        per_layer_rows.append({
            'layer': li,
            'k_median': np.nan,
            'v_median': np.nan,
            'k_p95': np.nan,
            'v_p95': np.nan,
            'median_ratio_k_over_v': np.nan,
            'n_finite_k': int(k_finite.size),
            'n_finite_v': int(v_finite.size),
        })
        continue

    k_norms.append(k_finite)
    v_norms.append(v_finite)

    per_layer_rows.append({
        'layer': li,
        'k_median': float(np.median(k_finite)),
        'v_median': float(np.median(v_finite)),
        'k_p95': float(np.percentile(k_finite, 95)),
        'v_p95': float(np.percentile(v_finite, 95)),
        'median_ratio_k_over_v': float(np.median(k_finite) / (np.median(v_finite) + 1e-8)),
        'n_finite_k': int(k_finite.size),
        'n_finite_v': int(v_finite.size),
    })

if not k_norms or not v_norms:
    raise RuntimeError('No finite K/V norms found. Cannot verify asymmetry.')

k_all = np.concatenate(k_norms)
v_all = np.concatenate(v_norms)

summary = pd.DataFrame([
    {
        'k_min': float(np.min(k_all)),
        'k_median': float(np.median(k_all)),
        'k_p95': float(np.percentile(k_all, 95)),
        'k_max': float(np.max(k_all)),
        'v_min': float(np.min(v_all)),
        'v_median': float(np.median(v_all)),
        'v_p95': float(np.percentile(v_all, 95)),
        'v_max': float(np.max(v_all)),
        'median_ratio_k_over_v': float(np.median(k_all) / (np.median(v_all) + 1e-8)),
        'p95_ratio_k_over_v': float(np.percentile(k_all, 95) / (np.percentile(v_all, 95) + 1e-8)),
        'input_tokens_used': int(inputs['input_ids'].shape[1]),
        'n_layers': len(layers),
        'layers_with_finite_stats': int(np.sum(np.isfinite(pd.DataFrame(per_layer_rows)['k_median']))),
    }
])

print('Overall K/V norm summary (finite values only):')
display(summary)

per_layer_df = pd.DataFrame(per_layer_rows)
print('Per-layer medians and ratios (first 10 rows):')
display(per_layer_df.head(10))

ratio = summary.loc[0, 'median_ratio_k_over_v']
if ratio > 5.0:
    print(f'Finding supported: key norms are substantially larger than value norms (median ratio={ratio:.2f}x).')
else:
    print(f'Finding weak in this run: median ratio={ratio:.2f}x; try longer context/repetitions.')

del outputs, model, inputs, cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Overall K/V norm summary (finite values only):


,k_min,k_median,k_p95,k_max,v_min,v_median,v_p95,v_max,median_ratio_k_over_v,p95_ratio_k_over_v,input_tokens_used,n_layers,layers_with_finite_stats
0,26.419884,735.736633,823.383545,823.906616,2.708874,4.183641,5.663383,6.664448,175.860367,145.387222,2048,28,2


Per-layer medians and ratios (first 10 rows):


,layer,k_median,v_median,k_p95,v_p95,median_ratio_k_over_v,n_finite_k,n_finite_v
0,0,778.002075,4.258503,823.623535,5.830011,182.693802,4096,4096
1,1,92.075325,4.076984,153.660355,5.531112,22.584173,2756,2756
2,2,NaN,NaN,NaN,NaN,NaN,0,0
3,3,NaN,NaN,NaN,NaN,NaN,0,0
4,4,NaN,NaN,NaN,NaN,NaN,0,0
5,5,NaN,NaN,NaN,NaN,NaN,0,0
6,6,NaN,NaN,NaN,NaN,NaN,0,0
7,7,NaN,NaN,NaN,NaN,NaN,0,0
8,8,NaN,NaN,NaN,NaN,NaN,0,0
9,9,NaN,NaN,NaN,NaN,NaN,0,0


Finding supported: key norms are substantially larger than value norms (median ratio=175.86x).


In [35]:
import gc
import time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from tqhk.cache import CacheConfig, TurboQuantDynamicCache

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
DEVICE = 'cuda'

NEEDLE = 'AURORA-7749'
QUESTION = 'What is the secret project code name? Answer exactly with the code only.'

FILLER = (
    'The quarterly review discussed staffing, infrastructure, milestones, and budget updates. '
    'Teams tracked dependencies, risks, and action items for cross-functional delivery.'
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'

def build_needle_prompt(tokenizer, target_tokens=1536):
    body = [f'The secret project code name is {NEEDLE}.']
    reps = 0
    prompt = ''
    n_tok = 0
    while reps < 400:
        user_content = '\n\n'.join(body + [QUESTION])
        messages = [
            {'role': 'system', 'content': 'You are a precise assistant.'},
            {'role': 'user', 'content': user_content},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        n_tok = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
        if n_tok >= target_tokens:
            break
        body.append(FILLER)
        reps += 1
    return prompt, n_tok

def run_needle_case(model, tokenizer, prompt, input_tokens, label, cache_cfg=None):
    cache = None
    if cache_cfg is not None:
        cache = TurboQuantDynamicCache(config=cache_cfg, n_layers=model.config.num_hidden_layers)

    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(DEVICE)

    start = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=24,
            do_sample=False,
            use_cache=True,
            past_key_values=cache,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.perf_counter() - start

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    exact = NEEDLE.lower() in response.lower()
    partial = ('aurora' in response.lower()) and ('7749' in response.lower())
    status = 'EXACT' if exact else ('PARTIAL' if partial else 'MISS')

    row = {
        'run_name': label,
        'status': status,
        'response_preview': response[:160],
        'latency_sec': elapsed,
        'input_tokens': int(input_tokens),
    }
    if cache is not None:
        stats = cache.get_stats()
        row['compression_token_fraction'] = stats.get('compression_token_fraction', 0.0)
        row['compressed_tokens'] = stats.get('compressed_tokens', 0)
        row['total_tokens'] = stats.get('total_tokens', 0)
    else:
        row['compression_token_fraction'] = 0.0
        row['compressed_tokens'] = 0
        row['total_tokens'] = 0

    return row

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.float16,
    attn_implementation='eager',
)
model.eval()

needle_prompt, input_tokens = build_needle_prompt(tok, target_tokens=1536)

cases = [
    ('baseline_fp16', None),
    ('tq_k8_v4_rw128', CacheConfig(key_bits=8, value_bits=4, residual_mode='fixed', residual_window=128)),
    ('tq_k6_v4_rw128', CacheConfig(key_bits=6, value_bits=4, residual_mode='fixed', residual_window=128)),
    ('tq_k4_v2_rw0', CacheConfig(key_bits=4, value_bits=2, residual_mode='fixed', residual_window=0)),
]

needle_rows = [run_needle_case(model, tok, needle_prompt, input_tokens, label, cfg) for label, cfg in cases]
needle_df = pd.DataFrame(needle_rows)
display(needle_df)

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

,run_name,status,response_preview,latency_sec,input_tokens,compression_token_fraction,compressed_tokens,total_tokens
0,baseline_fp16,EXACT,AURORA-7749,1.902741,1561,0.000000,0,0
1,tq_k8_v4_rw128,EXACT,AURORA-7749,3.217970,1561,0.918419,51876,56484
2,tq_k6_v4_rw128,EXACT,AURORA-7749,3.082886,1561,0.918419,51876,56484
3,tq_k4_v2_rw0,EXACT,"Based project name is A Aurora-7749. However, ...",9.379322,1561,1.000000,57024,57024


## 3) Reproduce TurboQuant V3 Needle Retrieval

> Goal: check whether our implementation preserves answer quality at moderate compression (for example K6/V4 with residual window) while aggressive configs fail.

We follow the same style as community tests: hide a code phrase in long context and ask model to recover it.

## 2) Verify K/V Norm Asymmetry (Community Finding #8)

This quick check runs a single cached forward pass and measures L2 norms of keys and values across layers/heads/tokens.
If key norms are consistently much larger than value norms, asymmetric K/V bit allocation is justified.

## 2) Primary Analysis: 3B Benchmark (Main Result)

This section is the canonical run for baseline vs TurboQuant cache tradeoffs.

It uses `Qwen/Qwen2.5-3B-Instruct` and reports refusal, KL, NLL delta, token disagreement, latency, and GPU or cache telemetry.

In [ ]:
cfg = BenchmarkConfig(
    model_name='Qwen/Qwen2.5-3B-Instruct',
    device='cuda',
    load_in_4bit=True,
    harmless_split='test[:20]',
    harmful_split='test[:20]',
    batch_size=4,
    kl_max_new_tokens=64,
    nll_target_new_tokens=32,
    max_new_tokens=96,
    filler_repetitions=32,
    output_csv='results/benchmark_results.csv',
    output_json='results/benchmark_results.json',
)

runs = [
    RunConfig(
        name='baseline_fp16_cache',
        use_turboquant_cache=False,
        cache_config=CacheConfig(),
    ),
    RunConfig(
        name='tq_k8_v4_rw128',
        use_turboquant_cache=True,
        cache_config=CacheConfig(
            key_bits=8,
            value_bits=4,
            residual_mode='fixed',
            residual_window=128,
        ),
    ),
    RunConfig(
        name='tq_k6_v4_rw128',
        use_turboquant_cache=True,
        cache_config=CacheConfig(
            key_bits=6,
            value_bits=4,
            residual_mode='fixed',
            residual_window=128,
        ),
    ),
    RunConfig(
        name='tq_k4_v2_rw128_prot2',
        use_turboquant_cache=True,
        cache_config=CacheConfig(
            key_bits=4,
            value_bits=2,
            residual_mode='fixed',
            residual_window=128,
            protected_layers=2,
            protected_bits=8,
        ),
    ),
]

rows = run_benchmark(cfg=cfg, run_configs=runs)
rows

README.md:   0%|          | 0.00/388 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/972k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/243k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25058 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6265 [00:00<?, ? examples/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [37]:
for r in rows:
    print(
        r['run_name'],
        'KL=', repr(r['avg_kl_to_baseline']),
        'NLL_delta=', repr(r['avg_nll_delta_to_baseline']),
        'token_disagreement=', repr(r['avg_token_disagreement_to_baseline']),
    )

baseline_fp16_cache KL= 0.0 NLL_delta= 0.0 token_disagreement= 0.0
tq_k8_v4_rw128 KL= 1.2749409675598145 NLL_delta= -0.5108394136652361 token_disagreement= 0.0328125
tq_k6_v4_rw128 KL= 2.861632823944092 NLL_delta= 0.007212315686047077 token_disagreement= 0.0828125
tq_k4_v2_rw32 KL= 3.885829210281372 NLL_delta= -2.204627928510309 token_disagreement= 0.2046875


In [ ]:
import pandas as pd
df = pd.read_csv('results/benchmark_results.csv')
cols = [
    'run_name',
    'refusal_rate',
    'avg_kl_to_baseline',
    'avg_nll_delta_to_baseline',
    'avg_token_disagreement_to_baseline',
    'avg_latency_sec',
    'cache_stats.memory_savings_ratio',
    'gpu_peak_allocated_mb',
    'gpu_peak_reserved_mb',
    'harmless_prompt_avg_tokens',
    'harmful_prompt_avg_tokens',
]
display(df[[c for c in cols if c in df.columns]])

,run_name,refusal_rate,avg_kl_to_baseline,avg_nll_delta_to_baseline,avg_token_disagreement_to_baseline,avg_latency_sec
0,baseline_fp16_cache,0.25,0.000000,0.000000,0.000000,3.973418
1,tq_k8_v4_rw128,0.25,1.274941,-0.510839,0.032813,25.222381
2,tq_k6_v4_rw128,0.25,2.861633,0.007212,0.082812,25.188878
3,tq_k4_v2_rw32,0.25,3.885829,-2.204628,0.204687,25.166469


## Interpretation of Latest Executed 3B Run

This interpretation is based on the currently displayed outputs (no re-run required).

- Flatline issue is resolved: KL and token disagreement are non-zero for TurboQuant runs and increase with more aggressive compression.
- Moderate compression (`tq_k8_v4_rw128`) shows the smallest drift from baseline.
- More aggressive compression (`tq_k6_v4_rw128`, `tq_k4_v2_rw32`) shows progressively higher divergence.
- Refusal rate stayed the same across runs in this sample (`0.25`), so the meaningful signal here is distributional drift, not refusal-rate change.
- Latency is currently much higher for TurboQuant cache runs than baseline in this implementation.

### Snapshot from the latest executed table

- baseline_fp16_cache: KL `0.000`, disagreement `0.0000`, latency `3.97s`
- tq_k8_v4_rw128: KL `1.275`, disagreement `0.0328`, latency `25.22s`
- tq_k6_v4_rw128: KL `2.862`, disagreement `0.0828`, latency `25.19s`
- tq_k4_v2_rw32: KL `3.886`, disagreement `0.2047`, latency `25.17s`

Practical takeaway: this notebook now demonstrates a clear compression-vs-fidelity tradeoff signal with 3B, while speedup is not reproduced in the current Python cache path.